# Scan for selection between populations (Fst), then view the sweep

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/06_popgen_selection.ipynb)

The compute→view loop on real data. Two *Drosophila melanogaster* populations — ancestral **African** and derived **cosmopolitan** — carry an insecticide-resistance allele that swept in the cosmopolitan range but not in Africa. Compute **Fst** from their allele frequencies and it peaks at *Cyp6g1*, right where the cosmopolitan population's diversity collapses. A differentiation peak sitting on a population-specific diversity valley is the signature of local adaptation — no single statistic proves it, their overlap does.

Frequencies are [DEST](https://dest.bio) Pool-Seq; the diversity bigWigs come from the same [population-genomics tutorial](https://jbrowse.org/jb2/docs/tutorials/population_genomics/#between-populations).

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Compute windowed Fst

Load the per-SNP African and cosmopolitan allele frequencies, then take Hudson Fst per 10 kb window (summed numerators over summed denominators). Swap the CSV for your own two frequency columns.

In [ ]:
import pandas as pd

freqs = pd.read_csv("https://jbrowse.org/demos/popgen/dest_cyp6g1_freqs.csv")
p1, p2 = freqs.afr_freq, freqs.cosmo_freq
freqs["num"] = (p1 - p2) ** 2                 # Hudson Fst numerator
freqs["den"] = p1 * (1 - p2) + p2 * (1 - p1)  # ... denominator
freqs["w"] = freqs.pos // 10_000 * 10_000

g = freqs.groupby("w")
windows = pd.DataFrame({
    "chrom": "chr2R",
    "start": g.size().index.astype(int),
    "end": g.size().index.astype(int) + 10_000,
    "fst": (g.num.sum() / g.den.sum()).clip(lower=0).round(3).values,
    "n_snps": g.size().values,
})
windows = windows[windows.n_snps >= 20]
windows.sort_values("fst", ascending=False).head()

## View the sweep on dm6

`fetch_hub("dm6")` pulls the fly genome, refName aliases, and a gene-name search index from the hosted hub. The computed Fst windows redden at the peak; the per-population diversity loads as a two-line wiggle — cosmopolitan collapses at the sweep while African holds.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, fetch_hub

BW = "https://jbrowse.org/demos/popgen/dest_cyp6g1_div_%s.bw"
div = lambda label, color, pop: {
    "type": "BigWigAdapter", "source": label, "color": color,
    "bigWigLocation": {"uri": BW % pop},
}

dm6 = fetch_hub("dm6")
view = LinearGenomeView(
    assembly=dm6["assemblies"][0],
    aggregate_text_search_adapters=dm6["aggregateTextSearchAdapters"],
    location="chr2R:11,900,000..12,450,000",  # or a gene name: "Cyp6g1"
)
view.add_features(
    windows,
    name="Fst (African vs cosmopolitan)",
    color="jexl:get(feature,'fst') > 0.25 ? '#d84315' : get(feature,'fst') > 0.12 ? '#f9a825' : '#90a4ae'",
)
view.add_track({
    "type": "MultiQuantitativeTrack",
    "trackId": "diversity",
    "name": "Nucleotide diversity (African vs cosmopolitan)",
    "adapter": {"type": "MultiWiggleAdapter", "subadapters": [
        div("African (ancestral)", "#377eb8", "african"),
        div("Cosmopolitan (derived)", "#e41a1c", "cosmopolitan"),
    ]},
    "displays": [{"type": "MultiLinearWiggleDisplay",
                  "displayId": "diversity-d", "defaultRendering": "multiline"}],
})
view.add_track(next(t for t in dm6["tracks"] if t["trackId"] == "dm6-ncbiRefSeqCurated"))
view